In [1]:
from IsingModel import Ising_Model as IM
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor

In [2]:
def single_run(args):
    S, T, iStep_Max = args

    model = IM(
        size=S,
        dTemperature=T,
        iStep_Max=iStep_Max
    )
    model.Run(record=False)

    return abs(model.CalculateMagnetization())

In [ ]:
def main():
    T_list = np.linspace(2.1, 2.4, 30)
    N_Run = 100
    Size_list = [80]
    iStep_Max = 5000
    max_workers = 32

    Tc_list = []
    time_list = []
    start = time.time()

    for S in Size_list:
        size_start = time.time()
        M_Avglist = []

        for T in T_list:
            tasks = [(S, T, iStep_Max) for _ in range(N_Run)]

            with ProcessPoolExecutor(max_workers=max_workers) as executor:
                M_Runs = list(executor.map(single_run, tasks))

            M_Avg = np.mean(M_Runs)
            M_Avglist.append(M_Avg)

        M_array = np.array(M_Avglist)
        T_array = np.array(T_list)

        slope = np.gradient(M_array, T_array)
        idx_critical = np.argmin(slope)
        T_critical = T_array[idx_critical]
        M_critical = M_array[idx_critical]

        size_elapsed = time.time() - size_start

        Tc_list.append(T_critical)
        time_list.append(size_elapsed)

        print("Size =", S)
        print("Estimated critical temperature =", T_critical)
        print("M at critical temperature =", M_critical)
        print("Minimum slope =", slope[idx_critical])
        print("Elapsed sec =", size_elapsed)
        print()

        df = pd.DataFrame({
            "Size": [S] * len(T_array),
            "T": T_array,
            "M": M_array
        })

        df.to_csv(
            "a.csv",
            mode="a",
            header=not os.path.exists("a.csv"),
            index=False
        )

        df_Tc = pd.DataFrame({
            "Size": [S],
            "Tc_estimated": [T_critical],
            "Elapsed_sec": [size_elapsed],
            "Elapsed_min": [size_elapsed / 60]
        })

        df_Tc.to_csv(
            "Tc.csv",
            mode="a",
            header=not os.path.exists("Tc.csv"),
            index=False
        )

    end = time.time()
    total_time = end - start
    print(f"time: {total_time} sec")
if __name__ == "__main__":
    main()